<a href="https://colab.research.google.com/github/KarunyaVarma/Karunya-Varma/blob/main/DNN_GTPP_Regression1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# -------------------------------------------------
# 1. Input data & Reproducibility
# -------------------------------------------------

tf.keras.utils.set_random_seed(42)

# Read the CSV file uploaded to Google Colab
data = pd.read_csv('/content/GTPP-DATASET.csv')

# 8 input features
X = data[[
    'AT',
    'AP',
    'DF',
    'CH',
    'GP',
    'CPR',
    'CPD',
    'TTXM'
]].values

# Target Output
y = data['EP'].values


# -------------------------------------------------
# 2. Train-Test Split (80:20 Ratio)
# -------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# -------------------------------------------------
# 3. MinMax Scaling
# -------------------------------------------------

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# -------------------------------------------------
# 4. (Phase-2)Settings & Epoch Strategies
# -------------------------------------------------

# Based on previous findings, we use the higher learning rate and smaller batch size
BEST_LR = 0.002   # Adjust to your exact highest-performing learning rate if higher
BEST_BS = 16      # Adjust to your exact best batch size (e.g., 16)
NEURONS = 128

# List of strategies to test: (max_epochs, use_early_stopping)
epoch_strategies = [
    (50, False),       # Baseline (50 epochs)
    (100, False),      # 100 epochs
    (200, False),      # 200 epochs
    (300, True)        # Up to 300 epochs with Early Stopping
]

results_list = []

print("Starting Epoch Experiments...\n")

# -------------------------------------------------
# 5. Loop Through Epoch Strategies
# -------------------------------------------------

for max_epochs, use_early_stop in epoch_strategies:

    # Create label for printout
    if use_early_stop == True:
        strategy_label = f"EarlyStop (Max {max_epochs})"
    else:
        strategy_label = f"Fixed {max_epochs}"

    print(f"Testing Strategy: {strategy_label} ...")

    # Reset seed for fair comparison
    tf.keras.utils.set_random_seed(42)

    # Build Model
    model = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(8,)),
        tf.keras.layers.Dense(NEURONS, activation='relu'),
        tf.keras.layers.Dense(1)
    ])

    # Compile Model
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=BEST_LR),
        loss='mse',
        metrics=['mae']
    )

    # Setup callbacks
    callbacks_list = []

    if use_early_stop == True:
        early_stop_callback = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=15,
            restore_best_weights=True
        )
        callbacks_list.append(early_stop_callback)

    # Train Model
    history = model.fit(
        X_train_scaled,
        y_train,
        epochs=max_epochs,
        batch_size=BEST_BS,
        validation_split=0.2,
        callbacks=callbacks_list,
        verbose=0
    )

    # Count how many epochs actually ran
    actual_epochs_run = len(history.history['loss'])

    # Predict on Test Data
    y_pred = model.predict(X_test_scaled, verbose=0).flatten()

    # Calculate Metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Store results
    results_list.append({
        "Epoch Strategy": strategy_label,
        "Actual Epochs Trained": actual_epochs_run,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    })


# -------------------------------------------------
# 6. Display Comparison Results
# -------------------------------------------------

results_dataframe = pd.DataFrame(results_list)

print("\n==================================================")
print("             EPOCH EXPERIMENT RESULTS             ")
print("==================================================")
print(results_dataframe.to_string(index=False))

Starting Epoch Experiments...

Testing Strategy: Fixed 50 ...
Testing Strategy: Fixed 100 ...
Testing Strategy: Fixed 200 ...
Testing Strategy: EarlyStop (Max 300) ...

             EPOCH EXPERIMENT RESULTS             
     Epoch Strategy  Actual Epochs Trained      MAE      MSE     RMSE  R2 Score
           Fixed 50                     50 1.240189 2.505994 1.583033  0.936151
          Fixed 100                    100 1.048892 1.872188 1.368279  0.952299
          Fixed 200                    200 1.025960 1.817833 1.348270  0.953684
EarlyStop (Max 300)                    122 1.044248 1.852782 1.361169  0.952794
